<a href="https://colab.research.google.com/github/erpanter/AdvancedTopicsInAI/blob/main/week4/3.fine-tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Fine-tuning

Another widely used transfer learning technique is _fine-tuning_.
Fine-tuning involves unfreezing a few of the top layers
of a frozen model base used for feature extraction, and jointly training both the newly added part of the model (in our case, the
fully-connected classifier) and these unfrozen top layers. This is called "fine-tuning" because it slightly adjusts the more abstract
representations of the model being reused, in order to make them more relevant for the problem at hand.

![fine-tuning VGG16](https://nyp-aicourse.s3.ap-southeast-1.amazonaws.com/it3103/resources/vgg16_fine_tuning.png)

In [1]:
import os
import keras
import keras.applications

It was necessary to freeze the convolution base of VGG16 in order to be able to train a randomly initialized
classifier on top. For the same reason, it is only possible to fine-tune the top layers of the convolutional base once the classifier on
top has already been trained. If the classified wasn't already trained, then the error signal propagating through the network during
training would be too large, and the representations previously learned by the layers being fine-tuned would be destroyed. Thus the steps
for fine-tuning a network are as follow:

1. Add your custom network on top of an already trained base network.
2. Freeze the base network.
3. Train the part you added.
4. Unfreeze some layers in the base network.
5. Jointly train both these layers and the part you added.


In [2]:
img_height, img_width = 128, 128

# Load the pre-trained model
base_model = keras.applications.VGG16(input_shape=(img_height, img_width) + (3,),
                                         include_top=False,
                                         weights='imagenet')

preprocess_input_fn = keras.applications.vgg16.preprocess_input

# freeze the base layer
base_model.trainable = False

# Add input layer
inputs = keras.layers.Input(shape=(img_height, img_width, 3))
# Add preprocessing layer
x = preprocess_input_fn(inputs)
# Add the base, set training to false to freeze the convolutional base
x = base_model(x)
# Add our classification head
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(rate=0.5)(x)
x = keras.layers.Dense(units=512, activation="relu")(x)
x = keras.layers.Dropout(rate=0.5)(x)
outputs = keras.layers.Dense(units=1, activation="sigmoid")(x)

model = keras.models.Model(inputs=[inputs], outputs=[outputs])

base_learning_rate = 0.001

model.compile(loss="binary_crossentropy",
                  optimizer=keras.optimizers.Adam(learning_rate=base_learning_rate),
                  metrics=["accuracy"])


58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Let's confirm all the layers of convolutional base are frozen.

In [3]:
for layer in base_model.layers:
    print(layer.name, layer.trainable)

input_layer False
block1_conv1 False
block1_conv2 False
block1_pool False
block2_conv1 False
block2_conv2 False
block2_pool False
block3_conv1 False
block3_conv2 False
block3_conv3 False
block3_pool False
block4_conv1 False
block4_conv2 False
block4_conv3 False
block4_pool False
block5_conv1 False
block5_conv2 False
block5_conv3 False
block5_pool False


Let's print out the model summary and see how many trainable weights. We can see that we only 263,169 trainable weights (parameters), coming from the classification head that put on top of the convolutional base. (For comparison, a VGG16 has total of 14,714,688 weights).

In [4]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 128, 128)  │          0 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 128, 128)  │          0 │ input_layer_1[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 128, 128)  │          0 │ input_layer_1[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 128, 128,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 128, 128,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg16 (Functional)  │ (None, 4, 4, 512) │ 14,714,688 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 512)       │          0 │ vgg16[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │    262,656 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 512)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │        513 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 14,977,857 (57.14 MB)

 Trainable params: 263,169 (1.00 MB)

 Non-trainable params: 14,714,688 (56.13 MB)

## Creating Datasets

We will setup our training and validation dataset as we did in earlier exercise.

In [5]:
dataset_URL = 'https://nyp-aicourse.s3-ap-southeast-1.amazonaws.com/datasets/cats_and_dogs_subset.tar.gz'
path_to_zip = keras.utils.get_file('cats_and_dogs_subset.tar.gz', origin=dataset_URL, extract=True, cache_dir='.')
dataset_dir = os.path.join(os.path.dirname(path_to_zip), "cats_and_dogs_subset_extracted/cats_and_dogs_subset")

67041740/67041740 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step


In [6]:
batch_size = 32
image_size = (img_height, img_width)

train_ds = keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="training",
    seed=1337,
    image_size=image_size,
    batch_size=batch_size,
    label_mode='binary'
)
val_ds = keras.utils.image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=1337,
    image_size=image_size,
    batch_size=batch_size,
    label_mode='binary'
)

Found 3000 files belonging to 2 classes.
Using 2400 files for training.
Found 3000 files belonging to 2 classes.
Using 600 files for validation.


## Train the classification head

We will go ahead and train our classification head.

In [7]:
# create model checkpoint callback to save the best model checkpoint
model_checkpoint_callback = keras.callbacks.ModelCheckpoint(
    filepath="best_checkpoint.weights.h5",
    save_weights_only=True,
    monitor='val_accuracy',
    mode='max',
    save_best_only=True)

model.fit(train_ds, validation_data=val_ds,
          epochs=30, callbacks=[model_checkpoint_callback])

Epoch 1/30


/usr/local/lib/python3.11/dist-packages/keras/src/models/functional.py:237: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: ['keras_tensor_19']
Received: inputs=Tensor(shape=(None, 128, 128, 3))
  warnings.warn(msg)


75/75 ━━━━━━━━━━━━━━━━━━━━ 25s 209ms/step - accuracy: 0.7836 - loss: 3.3332 - val_accuracy: 0.9283 - val_loss: 0.6232
Epoch 2/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 7s 86ms/step - accuracy: 0.9109 - loss: 1.1438 - val_accuracy: 0.9317 - val_loss: 0.3727
Epoch 3/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 6s 86ms/step - accuracy: 0.9155 - loss: 0.8452 - val_accuracy: 0.9400 - val_loss: 0.3734
Epoch 4/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 6s 84ms/step - accuracy: 0.9247 - loss: 0.4987 - val_accuracy: 0.9367 - val_loss: 0.2980
Epoch 5/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 7s 88ms/step - accuracy: 0.9187 - loss: 0.4242 - val_accuracy: 0.9417 - val_loss: 0.2084
Epoch 6/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 7s 87ms/step - accuracy: 0.9302 - loss: 0.4170 - val_accuracy: 0.9417 - val_loss: 0.2192
Epoch 7/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 6s 86ms/step - accuracy: 0.9350 - loss: 0.3165 - val_accuracy: 0.9350 - val_loss: 0.1921
Epoch 8/30
75/75 ━━━━━━━━━━━━━━━━━━━━ 7s 88ms/step - accuracy: 0.9392 - loss: 0.2491 - val_accuracy: 0.9483 - val_loss: 

In [8]:
model.load_weights('best_checkpoint.weights.h5')
eval_result = model.evaluate(val_ds)
print("[test loss, test accuracy]:", eval_result)

19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 68ms/step - accuracy: 0.9611 - loss: 0.1222
[test loss, test accuracy]: [0.12415929138660431, 0.9549999833106995]


## Fine-tuning last 3 convolutional layers and classification head together

We will go ahead and train our classification head.

Now we have our classification layers trained, let's start to unfreeze some top layers of the convolutional base to fine tune the weights.
We will fine-tune the last 3 convolutional layers, which means that all layers up until `block4_pool` should be frozen, and the layers
`block5_conv1`, `block5_conv2` and `block5_conv3` should be trainable.

Why not fine-tune more layers? Why not fine-tune the entire convolutional base? We could. However, we need to consider that:

* Earlier layers in the convolutional base encode more generic, reusable features, while layers higher up encode more specialized features. It is
more useful to fine-tune the more specialized features, as these are the ones that need to be repurposed on our new problem. There would
be fast-decreasing returns in fine-tuning lower layers.
* The more parameters we are training, the more we are at risk of overfitting. The convolutional base has 15M parameters, so it would be
risky to attempt to train it on our small dataset.

Thus, in our situation, it is a good strategy to only fine-tune the top 2 to 3 layers in the convolutional base.

Let's set this up, we will unfreeze our `base_model`,
and then freeze individual layers inside of it, except the last 3 layers.

Do a model ``summary()`` and you will see now that the number of trainable weights are now 7,079,424 (around 7 millions), much less than previously, because all the layers are frozen except the last 3 layers.

In [9]:
base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable = False

Let us examine model summary again. We can see now that we have more trainable weights 7,342,593 compared to previously 263,169.

In [10]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item (GetItem)  │ (None, 128, 128)  │          0 │ input_layer_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_1          │ (None, 128, 128)  │          0 │ input_layer_1[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ get_item_2          │ (None, 128, 128)  │          0 │ input_layer_1[0]… │
│ (GetItem)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stack (Stack)       │ (None, 128, 128,  │          0 │ get_item[0][0],   │
│                     │ 3)                │            │ get_item_1[0][0], │
│                     │                   │            │ get_item_2[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 128, 128,  │          0 │ stack[0][0]       │
│                     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ vgg16 (Functional)  │ (None, 4, 4, 512) │ 14,714,688 │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 512)       │          0 │ vgg16[0][0]       │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 512)       │          0 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 512)       │    262,656 │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 512)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │        513 │ dropout_1[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 15,504,197 (59.14 MB)

 Trainable params: 7,342,593 (28.01 MB)

 Non-trainable params: 7,635,264 (29.13 MB)

 Optimizer params: 526,340 (2.01 MB)

As you are training a much larger model and want to readapt the pretrained weights, it is important to use a lower learning rate at this stage as we do not want to make too drastic changes to the weights in the convolutional layers under fine-tuning.

In [11]:
finetune_learning_rate = base_learning_rate / 10.

model.compile(loss="binary_crossentropy",
              optimizer=keras.optimizers.Adam(learning_rate=finetune_learning_rate),
              metrics=["accuracy"])

model.fit(
    train_ds,
    epochs=15,
    validation_data=val_ds,
    callbacks=[model_checkpoint_callback])

Epoch 1/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 13s 112ms/step - accuracy: 0.9254 - loss: 0.2360 - val_accuracy: 0.9433 - val_loss: 0.1462
Epoch 2/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 17s 99ms/step - accuracy: 0.9481 - loss: 0.1500 - val_accuracy: 0.9500 - val_loss: 0.1688
Epoch 3/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 10s 100ms/step - accuracy: 0.9730 - loss: 0.0657 - val_accuracy: 0.9367 - val_loss: 0.1714
Epoch 4/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 12s 119ms/step - accuracy: 0.9785 - loss: 0.0638 - val_accuracy: 0.9433 - val_loss: 0.1424
Epoch 5/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 9s 102ms/step - accuracy: 0.9829 - loss: 0.0511 - val_accuracy: 0.9533 - val_loss: 0.1337
Epoch 6/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 10s 104ms/step - accuracy: 0.9891 - loss: 0.0354 - val_accuracy: 0.9583 - val_loss: 0.1270
Epoch 7/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 10s 100ms/step - accuracy: 0.9895 - loss: 0.0242 - val_accuracy: 0.9533 - val_loss: 0.2961
Epoch 8/15
75/75 ━━━━━━━━━━━━━━━━━━━━ 10s 99ms/step - accuracy: 0.9921 - loss: 0.0357 - val_accurac

In [12]:
model.load_weights('best_checkpoint.weights.h5')
eval_result = model.evaluate(val_ds)
print("[test loss, test accuracy]:", eval_result)

19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 69ms/step - accuracy: 0.9679 - loss: 0.1454
[test loss, test accuracy]: [0.1926925778388977, 0.9599999785423279]


**Exercise:**

1. Is our fine-tuned model performing better or worse?
2. Try to unfreeze less/more layers and see if the model performance improves.
